In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import lagrange
import package as pk

# Potential

https://linkinghub.elsevier.com/retrieve/pii/S0375960118309587


In [ ]:
def potential(q):
    return q**4/2 - q**2 + 0.5

def pdot(q):
    return - 2*q**3 + 2*q 
    

### Potential

In [ ]:
dx = 0.01     # resolution
x = np.arange(-2, 2, dx)
y = potential(x)
E = .25
# to compute the frequency it is needed the action at two neighbor values of E
E0 = E- E*0.001
E2 = E+ E*0.001


system = pk.Ham(potential, np.array([E0,E,E2]),x)

plt.plot(x,y)
plt.axhline(E, linestyle='dashed', color='red', linewidth=1)
plt.title('Effective Potential')
plt.xlabel('Radial distance')
plt.ylim(0,6)
plt.ylabel('Energy')

# Trajectory

In [ ]:
system.tol = 1e-7
system.add_pdot(pdot)
system.turns()
system.equilibrium()
system.get_regions()
system.get_trajectories()
system.get_actionfreq()
system.trajec.df()

# select trajectory
i=1
traj = system.trajec.df().loc[i]
traj

# Class Push

In [ ]:
class Push:
    def __init__(self):
        self.Nu = 100
        self.dt = 0.01
        self.step = 1000
        self.degree = 10

    def LF(self, system, i):
        x = [system.trajec.x1[i]]
        v = [system.p(system.trajec.x1[i], system.trajec.E[i])]
        if hasattr(system, 'pdot'):
            pdot = system.pdot
        else:
            pdot = lambda q: (system.V(q-system.tol) - system.V(q+system.tol))/(2*system.tol)
        for j in range(self.step):
            v_half = v[-1] + pdot(x[-1])*self.dt*0.5
            x_new = x[-1] + v_half*self.dt
            v_new = v_half + pdot(x_new)*self.dt*0.5
            x.append(x_new)
            v.append(v_new)        
        t = np.arange(0,self.step+1)*self.dt
        return t, x, v

    def SQ(self, system, i):
        ui = np.linspace(1, 2*np.pi, self.Nu)
        qi = pk.q_u(ui, system.trajec.x1[i], system.trajec.x2[i])
        abspi = system.p(qi, system.trajec.E[i])
        dt = (system.trajec.x2[i] - system.trajec.x1[i])/2*np.abs(np.sin(ui))/abspi
        dt[0] = dt[1]
        dt[-1] = dt[-2]
        trapez =  0.5*(dt[1]+dt[0])*(ui[1]-ui[0])
        ti = [ trapez ]
        for i in range(1,len(ui)-1):
            trapez = 0.5*(dt[i+1] + dt[i])*(ui[i+1]-ui[i])
            ti.append(ti[-1] + trapez)    
        ti = np.array(ti)
        ui = 0.5*(ui[1:] + ui[:-1])
        return ti, ui

    def PI(self, system, i):
        ui = np.linspace(1, 2*np.pi, self.Nu)
        qi = pk.q_u(ui, system.trajec.x1[i], system.trajec.x2[i])
        abspi = system.p(qi, system.trajec.E[i])
        dt = (system.trajec.x2[i] - system.trajec.x1[i])/2*np.abs(np.sin(ui))/abspi
        dt[0] = dt[1]
        dt[-1] = dt[-2]
        poly = np.polynomial.polynomial.Polynomial.fit(ui, dt, self.degree)
        t = poly.integ(k=0)
        ti = t(ui) - t(0)
        return ti, ui

## Evolve method for the Ham class

In [ ]:
system.pusher = Push()
def evol_traj(self, i, method):
    if method == 'LF':
        return system.pusher.LF(system,i)
    elif method == 'SQ':
        return system.pusher.SQ(system,i)
    elif method == 'PI':
        return system.pusher.PI(system,i)
    else:
        print('the method selected is not available. Chose between:\n - \'LF\' : Leap Frog $t,u,v$ \n - \'SQ\' : Simpson Quadrature (trapezoidal now) $t,u$ \n - \'PI\' : Polynomial integration $t,u$')

## test the package

### quadrature and polynomial integration

In [ ]:
t, u = system.evol_traj(i, 'SQ')
q = pk.q_u(u, system.trajec.x1[i], system.trajec.x2[i])
v = system.p(q, system.trajec.E[i])*np.sign(np.sin(u))
Ei = v**2/2 + system.V(q)

t1 = t
diffu = u
diffq = q
diffv = v
diffEi = Ei

fig, ax = plt.subplots(3,2, figsize=(14,18))
ax[0,0].plot(t,  u , label='change of coordinate u')
ax[0,0].plot(t, q, label='position' )
ax[0,0].plot(t, v , label='velocity' )
ax[0,0].legend()
ax[0,0].set_title('integration with quadrature')
ax[0,0].set_xlabel('$t$ - time')
#ax[0].set_xlim(t[0], t[-2])
ax[0,0].set_ylabel('dinamical variable')

ax[0,1].plot(t, Ei, label='Energy')
ax[0,1].set_xlabel('$t$ - time')
ax[0,1].set_ylabel('$E$ - energy')
ax[0,1].legend()
ax[0,0].set_ylabel('dinamical variable')

t, u = system.evol_traj(i, 'PI')
q = pk.q_u(u, system.trajec.x1[i], system.trajec.x2[i])
v = system.p(q, system.trajec.E[i])*np.sign(np.sin(u))
Ei = v**2/2 + system.V(q)

ax[1,0].plot(t,  u , label='change of coordinate u')
ax[1,0].plot(t, q, label='position' )
ax[1,0].plot(t, v , label='velocity' )
ax[1,0].legend()
ax[1,0].set_title('integration with polynomial fit')
ax[1,0].set_xlabel('$t$ - time')
ax[1,0].set_ylabel('dinamical variable')

ax[1,1].plot(t, Ei, label='Energy')
ax[1,1].set_xlabel('$t$ - time')
ax[1,1].set_ylabel('$E$ - energy')
ax[1,1].legend()
ax[1,0].set_ylabel('dinamical variable')

# plot the difference between the two

u = 0.5*(u[1:] + u[:-1])
q = pk.q_u(u, system.trajec.x1[i], system.trajec.x2[i])
v = system.p(q, system.trajec.E[i])*np.sign(np.sin(u))
Ei = v**2/2 + system.V(q)

diffu -= u
diffq -= q
diffv -= v
diffEi -= Ei

ax[2,0].plot(t1,  diffu , label='change of coordinate u')
ax[2,0].plot(t1, diffq, label='position' )
ax[2,0].plot(t1, diffv , label='velocity' )
ax[2,0].legend()
ax[2,0].set_title('difference between the solutions')
ax[2,0].set_xlabel('$t$ - time')
ax[2,0].set_ylabel('dinamical variable')

ax[2,1].plot(t1, diffEi, label='Energy')
ax[2,1].set_xlabel('$t$ - time')
ax[2,1].set_ylabel('$E$ - energy')
ax[2,1].legend()
ax[2,0].set_ylabel('dinamical variable')

### Leap Frog integrator

In [ ]:
t, r, v = system.evol_traj(i, 'LF')

fig, ax = plt.subplots(1,2, figsize=(14,6))
ax[0].plot(t, r, label='position' )
ax[0].plot(t, v , label='velocity' )
ax[0].legend()
fig.suptitle('integration with polynomial fit')
ax[0].set_xlabel('$t$ - time')
ax[0].set_xlim(0, t[-1])
ax[0].set_ylabel('dinamical variable')

Ei =  [ 0.5*v[i]*v[i] + potential(r[i]) for i in range(len(t)) ]
ax[1].plot(t, Ei, label='Energy')
ax[1].set_xlabel('$t$ - time')
ax[1].set_ylabel('$E$ - energy')
ax[1].legend()
ax[0].set_ylabel('dinamical variable')


# Test of Numerical Methods

#### Change of variable
it is a utility function in the package

    pk.q_u(u,q0,q1)

## Phase potrait

In [ ]:
ui = np.linspace(0,2*np.pi, 100, endpoint=True )

qi = pk.q_u(ui, traj['x1'], traj['x2'])
absvi = system.p(qi, traj['E'])



# definition of the velocity through the conservation of Energy 
fig, axs = plt.subplots(1,2, figsize=(14,7))

axs[0].scatter(ui, np.sign(np.sin(ui))*absvi, label='$q(u)$')
axs[0].legend()
axs[0].set_ylabel('Radial velocity')
axs[0].set_xlabel('Angle in phase space')
axs[0].set_title('Phase potrait')


axs[1].scatter(qi, np.sign(np.sin(ui))*absvi, label='Energy level {:.2f}'.format(traj['E']))
    
axs[1].legend()
axs[1].set_title('Phase potrait')
axs[1].set_xlabel('Radial distance')
axs[1].set_ylabel('Radial velocity')
plt.show()

## Numerical quadrature approach

### Integrand

In [ ]:
f = np.sin(ui)/absvi*np.sign(np.sin(ui))
f = (traj['x2'] - traj['x1'])/2*f

f[0] = f[1]
f[-1] = f[-2]

# plt.plot(ui, f)

In [ ]:
trapez =  0.5*(f[1]+f[0])*(ui[1]-ui[0])
t = [ trapez ]
for i in range(1,len(ui)-1):
    trapez = 0.5*(f[i+1] + f[i])*(ui[i+1]-ui[i])
    t.append(t[-1] + trapez)

t = np.array(t)

### Trapezoidal Quadrature

In [ ]:
u = 0.5*(ui[1:]+ui[:-1])
q = pk.q_u(u, traj['x1'], traj['x2'])
v = system.p(q, traj['E'])*np.sign(np.sin(u))
Ei = v**2/2 + potential(q)

fig, ax = plt.subplots(1,2, figsize=(14,6))
ax[0].plot(t,  u , label='change of coordinate u')
ax[0].plot(t, q, label='position' )
ax[0].plot(t, v , label='velocity' )
ax[0].legend()
fig.suptitle('integration with polynomial fit')
ax[0].set_xlabel('$t$ - time')
ax[0].set_xlim(t[0], t[-1])
ax[0].set_ylabel('dinamical variable')

ax[1].plot(t, Ei, label='Energy')
ax[1].set_xlabel('$t$ - time')
ax[1].set_ylabel('$E$ - energy')
ax[1].legend()
ax[0].set_ylabel('dinamical variable')


### polynomial interpolation and integration

In [ ]:
degree = 10
poly = np.polynomial.polynomial.Polynomial.fit(ui,f, degree)

t = poly.integ(k=0)
ti = t(ui) - t(0)

In [ ]:
Ei = absvi**2/2 + potential(qi)

fig, ax = plt.subplots(1,2, figsize=(14,6))
ax[0].plot(ti,  ui , label='change of coordinate u')
ax[0].plot(ti, qi, label='position' )
ax[0].plot(ti, np.sign(np.sin(ui))*absvi , label='velocity' )
ax[0].legend()
fig.suptitle('integration with polynomial fit')
ax[0].set_xlabel('$t$ - time')
ax[0].set_xlim(ti[0], ti[-1])
ax[0].set_ylabel('dinamical variable')

ax[1].plot(ti, Ei, label='Energy')
ax[1].set_xlabel('$t$ - time')
ax[1].set_ylabel('$E$ - energy')
ax[1].legend()
ax[0].set_ylabel('dinamical variable')


## Compare with standard integration methods: RK and leapfrog

In [ ]:
def pdot(q):
    return - 2*q**3 +  2*q

def rdot(r,E):
    return np.real(np.sqrt( 2*E - 2*potential(r), dtype='complex' ))

### RK4 -- _Errors in the implementation_


 \begin{align}\frac{dy}{dt} = f(y), \quad y(t_0) = y_0.\end{align}

\begin{align}
 k_1 &= \ f(t_n, y_n), \\
 k_2 &= \ f\!\left(t_n + \frac{h}{2}, y_n + h \frac{k_1}{2}\right), \\ 
 k_3 &= \ f\!\left(t_n + \frac{h}{2}, y_n + h \frac{k_2}{2}\right), \\
 k_4 &= \ f\!\left(t_n + h, y_n + h k_3\right).
\end{align}

\begin{align}
y_{n+1} &= y_n + \frac{h}{6}\left(k_1 + 2k_2 + 2k_3 + k_4 \right),\\
t_{n+1} &= t_n + h \\
\end{align}

In [ ]:
def rk4(r, E):
    k1 = rdot(r, E)
    k2 = rdot(r + dt*k1/2, E)
    k3 = rdot(r + dt*k2/2, E)
    k4 = rdot(r + dt*k3, E)
    r_new = r + dt/6*(k1 + 2*k2 + 2*k3 + k4)
    return r_new

#### Evolution

In [ ]:


step = 1000          # lenght of the evolution
dt = 6*np.pi/step           # resolution

r = np.zeros(step)
r[0] = 0
for i in range(1,len(r)):
    t = dt*i
    r[i] = rk4(r[i-1], E)

### LF - Leapfrog



\begin{align}
  \dot{q} = \partial_p H  = p \\
  \dot{p} = - \partial_q H = - \partial_q V(q) \\
\end{align}

LEAP FROG
\begin{align}
  p_{i+1/2} &= p_{i-1/2} + \dot{p}(q_{i})\, \Delta t, \\
  q_{i+1} &= q_{i} + p_{i+1/2}\, \Delta t,
\end{align}

SEMI-LAGRANGIAN STYLE
\begin{align}
  p_{i+1/2} &= p_i + \dot{p}(q_i) \frac{\Delta t}{2}, \\
  q_{i+1} &= q_i + p_{i+1/2}\Delta t,\\
  p_{i+1} &= p_{i+1/2} + \dot{p}(q_{i+1}) \frac{\Delta t}{2},
\end{align}

In [ ]:
def SM_LF(r, v, dt):
    v_half = v + pdot(r)*dt*0.5
    r_new = r + v_half*dt
    v_new = v_half + pdot(r_new)*dt*0.5
    return r_new, v_new

In [ ]:
def LF(r, v, dt):
    v_new = v + pdot(r)*dt
    r_new = r + v_new*dt
    return r_new, v_new

#### Evolution

In [ ]:
###########################
step = 1000
dt = .01
###########################

r = np.zeros(step)
v = rdot(r,E)
r[0] = traj['x1']
for i in range(1,len(r)):
    t = dt*i
    r[i], v[i] = SM_LF(r[i-1], v[i-1], dt)

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(14,6))
ax[0].plot(dt*np.arange(0,len(r)), r, label='position' )
ax[0].plot(dt*np.arange(0,len(r)), v , label='velocity' )
ax[0].legend()
fig.suptitle('integration with polynomial fit')
ax[0].set_xlabel('$t$ - time')
ax[0].set_xlim(0, len(r)*dt)
ax[0].set_ylabel('dinamical variable')

Ei =  0.5*v*v + potential(r)
ax[1].plot(dt*np.arange(0,len(r)), Ei, label='Energy')
ax[1].set_xlabel('$t$ - time')
ax[1].set_ylabel('$E$ - energy')
ax[1].legend()
ax[0].set_ylabel('dinamical variable')


In [ ]:
plt.scatter(r,v, s=1)
plt.plot(qi, np.sign(np.sin(ui))*absvi, linestyle='dotted', color='black', linewidth=1)
plt.xlabel('q')
plt.ylabel('p')